## Load RNN and compare against HMM at forecasting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import rpy2.robjects as robjects

In [ ]:
from custom_losses import (
    dirichlet_layer,
    EvidentialLoss,
    evidential_kl_divergence,
    cross_entropy_loss,
)

In [ ]:
readRDS = robjects.r['readRDS']

def load_rds_array(path):
    """Load an R array into numpy, preserving shape and dimnames."""
    r_obj    = readRDS(path)
    dims     = tuple(int(d) for d in robjects.r['dim'](r_obj))
    arr      = np.array(r_obj).reshape(dims, order='F')
    r_dimnames = robjects.r['dimnames'](r_obj)
    dimnames = []
    for dn in r_dimnames:
        if dn == robjects.rinterface.NULL:
            dimnames.append(None)
        else:
            dimnames.append(list(dn))
    return arr, dimnames

X_train,     (_, _, feature_names)   = load_rds_array('energy_data/X_train.rds')
Y_train,     (_, _, state_names)     = load_rds_array('energy_data/Y_train.rds')
Y_hat_train, _                       = load_rds_array('energy_data/Y_hat_train.rds')

X_valid,     _ = load_rds_array('energy_data/X_valid.rds')
Y_valid,     _ = load_rds_array('energy_data/Y_valid.rds')
Y_hat_valid, _ = load_rds_array('energy_data/Y_hat_valid.rds')

X_test,      _ = load_rds_array('energy_data/X_test.rds')
Y_test,      _ = load_rds_array('energy_data/Y_test.rds')
Y_hat_test,  _ = load_rds_array('energy_data/Y_hat_test.rds')

print("X_train:    ", X_train.shape)
print("Y_train:    ", Y_train.shape)
print("Feature names:", feature_names)
print("State names:  ", state_names)

In [ ]:
# HMM transition matrix
transition_matrix = pd.read_csv('energy_data/transition_matrix.csv', index_col=0)
transition_matrix.columns = range(len(transition_matrix.columns))
transition_matrix.index   = range(len(transition_matrix.index))
print("Transition matrix:")
print(transition_matrix)

In [ ]:
best_model = tf.keras.models.load_model('energy_data/best_model.keras')
best_model.summary()